In [ ]:
# !pip install cudaq==0.11.0

In [ ]:
import numpy as np
import pandas as pd
import cudaq
from cudaq import spin
import sys
import os
import torch
from typing import List, Tuple
from torch.optim import Adam, AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, CosineAnnealingWarmRestarts, ReduceLROnPlateau, ExponentialLR, CyclicLR, SequentialLR
from tqdm import tqdm
from math import sqrt
import matplotlib.pyplot as plt

# Feel free to modify the following parameters as needed

### Objective of this problem is to
$$
\max_{\mathbf{x}} \left\{ \mu^\top \mathbf{x} - q \mathbf{x}^\top C \mathbf{x}\right\} \\
\text{s.t. } \sum_{i=1}^n x_i = 1
$$
where 
- $\mathbf{x}$ is the decision variable
- $\mu$ is the expected return vector
- $C$ is the covariance matrix of returns
- q is the risk aversion parameter, $P$ is the price vector
- $\lambda$ is a penalty parameter for budget constraint violation.

In [ ]:
seed = 55
dataset_dir = "./dataset/"
budget = 1000
lamb = 0.0005
q = 1.5
n_assets = 4
layer_count = 5
hamiltonian_boost = 5000
is_pbar = True

np.random.seed(seed)
torch.manual_seed(seed)
device = torch.device("cuda:0")
cudaq.set_target("nvidia")

data_cov_pd = pd.read_csv(os.path.join(dataset_dir, "top_50_us_stocks_data_20250526_011226_covariance.csv"))
data_ret_p_pd = pd.read_csv(os.path.join(dataset_dir, "top_50_us_stocks_returns_price.csv"))

asset_idx = np.random.choice(data_cov_pd.shape[0], n_assets, replace=False)
data_cov = data_cov_pd.drop("Ticker", axis=1).to_numpy()[asset_idx, :][:, asset_idx]
stock_names = data_ret_p_pd["Company_Name"].to_numpy()[asset_idx]
# print("Selected Stocks: ", stock_names.tolist())
data_ret_p = data_ret_p_pd.drop("Ticker", axis=1)
asset_idx_raw = data_ret_p.index[asset_idx].to_numpy()
data_ret_p = data_ret_p.drop("Company_Name", axis=1).to_numpy()[asset_idx, :]
data_ret = data_ret_p[:, 0]
data_p = data_ret_p[:, 1]

for i in range(n_assets):
    print(f"Stock: {stock_names[i]}, Return: {data_ret[i]:.4f}, Price: {data_p[i]:.2f}")

P = data_p
ret = data_ret
cov = data_cov
B = budget
parameter_count = 2 * layer_count

### Normalize problem into
$$
\max_{\mathbf{x}} \left\{ \mu'^\top \mathbf{n} - q \mathbf{n}^\top C' \mathbf{n}  \right\} \\ %% -\lambda (P'^\top \mathbf{n} - 1)^2
\text{s.t. } P'^\top \mathbf{n} = 1
$$
where
\begin{align*}
P' &= \frac{P}{B}\\
\mu' &= \text{diag}(P') \mu\\
C' &= \text{diag}(P') C \text{diag}(P')
\end{align*}

### Encode the problem into binary variables

number of qubits needed for each asset $i$ is given by
$$
d_i = \lfloor \log_2(n_i + 1) \rfloor + 1
$$
Accordingly, the total number of qubits needed is
$$
n_{\text{qb}} = \sum_{i=1}^N d_i
$$
Define transition matrix $E$ map binary variables to the original variables as
$$
E=\begin{pmatrix}
        2^0 & \cdots & 2^{d_1-1} & 0 & \cdots & 0 & \cdots & 0 & \cdots & 0 \\
        0 & \cdots & 0 & 2^0 & \cdots & 2^{d_2-1} & \cdots & 0 & \cdots & 0 \\
        \vdots & \ddots & \vdots & \vdots & \ddots & \vdots & \ddots & \vdots & \ddots & \vdots \\
        0 & \cdots & 0 & 0 & \cdots & 0 & \cdots & 2^0 & \cdots & 2^{d_N-1}
    \end{pmatrix}
$$

By introducing a penalty term for the budget constraint, the problem can be rewritten as
$$
\max_{\mathbf{n}} \left\{ \mu''^\top \mathbf{v} - q \mathbf{v}^\top C'' \mathbf{v} - \lambda (P''^\top \mathbf{v} - 1)^2 \right\} \\
\text{s.t. } P''^\top \mathbf{v} = 1
$$

where
\begin{align*}
P'' &= E^\top P'\\
\mu'' &= E^\top \mu'\\
C'' &= E^\top C' E
\end{align*}

In [ ]:
def po_normalize(B, P, ret, cov):
    P_b = P / B
    ret_b = ret * P_b
    cov_b = np.diag(P_b) @ cov @ np.diag(P_b)
    
    n_max = np.int32(np.floor(np.log2(B/P))) + 1
    # print("n_max:", n_max)
    n_qs = np.cumsum(n_max)
    n_qs = np.insert(n_qs, 0, 0)
    n_qubit = n_qs[-1]
    E = np.zeros((len(P), n_qubit))
    for i in range(len(P)):
         for j in range(n_max[i]):
              E[i, n_qs[i] + j] = 2**j

    P_bb = E.T @ P_b
    ret_bb = E.T @ ret_b
    cov_bb = E.T @ cov_b @ E
    return P_bb, ret_bb, cov_bb, int(n_qubit), n_max, E

P_bb, ret_bb, cov_bb, n_qubit, n_max, E = po_normalize(B, P, ret, cov)
print("n_qubit:", n_qubit)
print("n_max:", n_max)

### QUBO formulation
We can now obtain the QUBO formulation of the problem as
$$
QU = \text{diag}(\mu'' + 2\lambda P'') - q C'' - 2\lambda P'' P''^\top
$$

and problem can rewritten as
$$
\max_{\mathbf{v}} \left\{ \mathbf{v}^\top QU \mathbf{v} \right\} \\
\text{s.t. } P''^\top \mathbf{v} = 1
$$

In [ ]:
def ret_cov_to_QUBO(ret: np.ndarray, cov: np.ndarray, P: np.ndarray, lamb: float, q:float) -> np.ndarray: # Max return, Min variance
    di = np.diag(ret + 2*lamb*P)
    mat = lamb * np.outer(P, P) + q * cov
    return di - mat

QU = ret_cov_to_QUBO(ret_bb, cov_bb, P_bb, lamb, q)
QU_eval = ret_cov_to_QUBO(ret_bb, cov_bb, P_bb, 0.0, q)
QU_lamb = ret_cov_to_QUBO(np.zeros_like(ret_bb), np.zeros_like(cov_bb), P_bb, lamb, q)

### Ising formulation
Consider that for each element of $QU_{ij}$ can be represented as
$$
H(QU) = \sum_{ii} QU_{ii} \frac{I - Z_i}{2} + \sum_{i\neq j} QU_{ij} \frac{I - Z_i}{2} \otimes \frac{I - Z_j}{2}
$$

In [ ]:
def qubo_to_ising(qubo: np.ndarray, lamb: float) -> cudaq.SpinOperator:
    spin_op = -lamb * spin.i(0)
    for i in range(qubo.shape[0]):
        for j in range(qubo.shape[1]):
                if i != j and qubo[i, j] != 0:
                    # Problem unitary of Pauli String of Z_i Z_j
                    # Use '*' for tensor product of Pauli operators (e.g. Z0Z1 -> spin.z(0) * spin.z(1))
                    ### Put your code here ###
                    spin_op += 
                    ##########################
                elif i == j and qubo[i, j] != 0:
                    # Problem unitary of Pauli String of Z_i
                    ### Put your code here ###
                    spin_op += 
                    ##########################
    return spin_op

H = -qubo_to_ising(QU, lamb).canonicalize() * hamiltonian_boost
H_eval = -qubo_to_ising(QU_eval, 0.0).canonicalize() * hamiltonian_boost
H_lamb = -qubo_to_ising(QU_eval, lamb).canonicalize() * hamiltonian_boost

Quantum state of $n_\text{qb}$ qubits $|\psi\rangle$ can be obtain by $L$ layer QAOA circuit formulated as
$$
\ket{\psi(\boldsymbol{\beta},\boldsymbol{\gamma})}
    =
    \prod_{\ell=1}^{L}
    e^{-i\beta_\ell H_X}
    e^{-i\gamma_\ell H(\lambda)}
    \ket{\psi_0},
$$
where $H_X = \sum_{i=1}^{n_\text{qb}} X_i$ is the mixer Hamiltonian and $\ket{\psi_0} = |+\rangle^{\otimes n_\text{qb}}$ is the initial state.

Now, problem can viewed as adiabatic evolution from $H_X$ to $H(\lambda)$, which can approximate the optimal solution by optimizing the parameters $\boldsymbol{\beta}$ and $\boldsymbol{\gamma}$ to minimize the expectation value of $H(\lambda)$:
$$
\min_{\boldsymbol{\beta},\boldsymbol{\gamma}} \langle \psi(\boldsymbol{\beta},\boldsymbol{\gamma}) | H(\lambda) | \psi(\boldsymbol{\beta},\boldsymbol{\gamma}) \rangle
$$

Consider that our hamiltonian is of the form
$$
H = \sum_{i} h_i Z_i + \sum_{i<j} J_{ij} Z_i Z_j
$$
We can trotterize and implement the unitary evolution $e^{-i\gamma H}$ as
$$
e^{-i\gamma H} \approx \prod_{i} e^{-i\gamma h_i Z_i} \prod_{i<j} e^{-i\gamma J_{ij} Z_i Z_j}
$$


Note that the trotterization error can derive into
$$
e^{tA}e^{tB} \approx e^{t(A+B)} + \frac{t^2}{2} [A, B] + O(t^3)

In [ ]:
# Obtain coefficients and indices for the cost Hamiltonian h_i and J_if terms
def process_ansatz_values(H: cudaq.SpinOperator) -> Tuple[List[int], List[float], List[int], List[int], List[float]]:
    HH = H.get_raw_data()
    idxs = [[j - len(HH[0][i])//2 for j in range(len(HH[0][i])) if HH[0][i][j]] for i in range(len(HH[0]))]

    HH = [(idxs[i], HH[1][i], sum(HH[0][i])) for i in range(len(HH[0]))]
    HH = sorted(HH, key=lambda x: (x[2], x[0]), reverse=False)

    idx_1 = []
    coeff_1 = []
    idx_2_a, idx_2_b = [], []
    coeff_2 = []
    for i in range(len(HH)):
        if HH[i][1].real == 0:
            continue
        if HH[i][2] == 1:
            idx_1.append(HH[i][0][0])
            coeff_1.append(HH[i][1].real)
        elif HH[i][2] == 2:
            idx_2_a.append(HH[i][0][0])
            idx_2_b.append(HH[i][0][1])
            coeff_2.append(HH[i][1].real)

    return idx_1, coeff_1, idx_2_a, idx_2_b, coeff_2

idx_1_use, coeff_1_use, idx_2_a_use, idx_2_b_use, coeff_2_use = process_ansatz_values(H)
coeff_1_use, coeff_2_use = np.array(coeff_1_use), np.array(coeff_2_use)
ansatz_fixed_param = (int(n_qubit), layer_count, idx_1_use, coeff_1_use, idx_2_a_use, idx_2_b_use, coeff_2_use)

In [ ]:
@cudaq.kernel
def kernel_qaoa_X(thetas: List[float], qubit_count: int, layer_count: int, idx_1: List[int], coeff_1: List[float], idx_2_a: List[int], idx_2_b: List[int], coeff_2: List[float]):
    '''
    thetas: List of parameters for the QAOA circuit. The first `layer_count` elements are for the cost unitary, and the next `layer_count` elements are for the mixer unitary.
    qubit_count: Total number of qubits in the circuit.
    layer_count: Number of layers in the QAOA circuit.
    idx_1: List of qubit indices for the single-qubit Z rotations corresponding to the linear terms in the cost Hamiltonian.
    coeff_1: List of coefficients for the single-qubit Z rotations corresponding to the linear terms in the cost Hamiltonian.
    idx_2_a: List of qubit indices for the first qubit in the two-qubit interactions corresponding to the quadratic terms in the cost Hamiltonian.
    idx_2_b: List of qubit indices for the second qubit in the two-qubit interactions corresponding to the quadratic terms in the cost Hamiltonian.
    coeff_2: List of coefficients for the two-qubit interactions corresponding to the quadratic terms in the cost Hamiltonian.
    '''
    qreg = cudaq.qvector(qubit_count)
    # create uniform superposition
    h(qreg)

    for i in range(layer_count):

        for j in range(len(idx_1)):
            rz(2 * coeff_1[j] * thetas[i], qreg[idx_1[j]])
        
        for j in range(len(idx_2_a)):
            # Problem unitary of Pauli String of Z_i Z_j
            ### Put your code here ###
            
            ##########################

        for j in range(qubit_count):
            # Mixer unitary with Pauli X
            ### Put your code here ###
            
            ##########################

@cudaq.kernel
def kernel_flipped(state: cudaq.State, n_qb: int):
    q = cudaq.qvector(state)
    for i in range(n_qb//2):
        swap(q[i], q[n_qb - 1 - i])

print(cudaq.draw(kernel_qaoa_X, [0.5]*n_qubit, int(n_qubit), 1, idx_1_use, coeff_1_use, idx_2_a_use, idx_2_b_use, coeff_2_use))

In [ ]:
points = np.random.uniform(-0.01, 0.01, (2*layer_count))
max_iter = 300
SHIFT = 1e-4
F_TOL = 1e-4
points_cu = torch.tensor(points, dtype=torch.float64, device=device)
optimizer_cu = Adam([points_cu], lr=0.01, betas=(0.95, 0.98), weight_decay=0.01, decoupled_weight_decay=True)
scheduler_co = CosineAnnealingLR(optimizer_cu, T_max=max_iter, eta_min=0.0003)
scheduler_cu = ExponentialLR(optimizer_cu, gamma=0.987)
scheduler_warmup = CyclicLR(optimizer_cu, base_lr=0.01, max_lr=0.012, step_size_up=10, step_size_down=10, mode='triangular2')
# scheduler_all = SequentialLR(optimizer_cu, schedulers=[scheduler_warmup, scheduler_cu], milestones=[40])
# scheduler_all = SequentialLR(optimizer_cu, schedulers=[scheduler_warmup, scheduler_co], milestones=[40])
scheduler_all = scheduler_co

In [ ]:
optimal_expectation, optimal_parameters = None, None
expectations = []
pbar_optim = tqdm(range(max_iter), disable=not is_pbar)
num_iter = 0
last_f = None
for it in pbar_optim:
    optimizer_cu.zero_grad()
    params = points_cu.detach().clone()
    expectation = float(cudaq.observe(kernel_qaoa_X, H, params.cpu().numpy(), *ansatz_fixed_param).expectation())
    num_iter += 1
    expectation_eval = float(cudaq.observe(kernel_qaoa_X, H_eval, params.cpu().numpy(), *ansatz_fixed_param).expectation())

    expectation_lamb = float(cudaq.observe(kernel_qaoa_X, H_lamb, params.cpu().numpy(), *ansatz_fixed_param).expectation()) / hamiltonian_boost
    expectation_violate = sqrt(expectation_lamb / lamb)
    grad = torch.zeros_like(params)
    # print(grad.dtype)
    for j in range(parameter_count):
        shift = np.zeros(parameter_count)
        shift[j] = SHIFT
        forward = float(cudaq.observe(kernel_qaoa_X, H, (params.cpu().numpy() + shift), *ansatz_fixed_param).expectation())
        # backward = float(cudaq.observe(kernel_qaoa_X, H, (params.cpu().numpy() - shift), *ansatz_fixed_param).expectation())
        # grad[j] = (forward - backward) / (2.0 * SHIFT)
        grad[j] = (forward - expectation) / SHIFT
    points_cu.grad = grad
    optimizer_cu.step()
    # scheduler_cu.step()
    # scheduler_cu.step(expectation)
    scheduler_all.step()
    expectations.append([expectation/hamiltonian_boost, expectation_eval/hamiltonian_boost, expectation_lamb, points_cu[0].item(), points_cu[1].item()])
    
    cou_con = cou_con + 1 if last_f is not None and abs(expectation - last_f) < F_TOL else 0
    if cou_con >= 3:
        break
    last_f = expectation
    
    if is_pbar:
        pbar_optim.set_description(f"Iter {it}, Exp_obj {expectation/hamiltonian_boost:.6f}, Exp_eval {expectation_eval/hamiltonian_boost:.6f}, Exp_lamb {expectation_lamb:.6f}, LR {optimizer_cu.param_groups[0]['lr']:.4f}")
optimal_parameters = points_cu.cpu().numpy()

In [ ]:
expectations = np.array(expectations)
plt.figure(figsize=(10, 6))
energy_obj = expectations[:, 0]
energy_eval = expectations[:, 1]
energy_lamb = expectations[:, 2]
plt.plot(energy_obj, label="Objective Energy")
plt.plot(energy_eval, label="Eval Energy")
plt.plot(energy_lamb, label="Lambda Energy")
plt.xlabel("Iteration")
plt.ylabel("Energy")
plt.title("Energy vs Iteration")
plt.legend()
plt.grid()
plt.show()

In [ ]:
result = cudaq.get_state(kernel_qaoa_X, optimal_parameters, *ansatz_fixed_param)
idx_r_bests = np.argsort(np.abs(result))[-5:][::-1]
idx_bests = [bin(idx_r)[2:].zfill(n_qubit)[::-1] for idx_r in idx_r_bests]

def all_state_to_return(qb, lam, QUBO): # QUBO of Max problem
    '''
    IMPORTANT: 
        QUBO: must be QUBO of MAX problem!!!
    '''
    ll = np.zeros((qb, 1<<qb), dtype=np.float32)
    a_0 = np.zeros(1<<qb, dtype=np.float32)
    a_1 = np.ones(1<<qb, dtype=np.float32)
    idxx = np.arange(1<<qb, dtype=np.int32)
    for i in range(qb):
        ll[i] = np.where(idxx%(1<<(qb-i))<(1<<(qb-i-1)), a_0,  a_1)
    l = ll.T.copy()
    ss = l @ QUBO
    ss = (ss.reshape(-1, 1, qb) @ l.reshape(-1, qb, 1))
    return ss.reshape(-1) - lam
state_optim = all_state_to_return(n_qubit, lamb, QU)
state_eval = all_state_to_return(n_qubit, 0.0, QU_eval)

result_r = cudaq.get_state(kernel_flipped, result, n_qubit)
prob = np.abs(result_r)**2
optimal_expectation = (prob * (state_eval)).sum()
print("Expected value:", optimal_expectation)

for i, idx_best in enumerate(idx_bests):
    print(f"\nOptimal binary string #{i+1}:", idx_best)
    vec_best = np.array([int(x) for x in idx_best])
    n_buy = E @ vec_best
    print("Number of each asset to buy:", n_buy)
    print("Budget used:", round(np.sum(n_buy * P), 2))
    print("Expected value: ", round(state_optim[int(idx_best, 2)], 7))
    print("Expected return:", round(state_eval[int(idx_best, 2)], 7))

In [ ]:
idx_best_evals = np.argsort(state_optim)[-5:][::-1]
for i, idx_best_eval in enumerate(idx_best_evals):
    idx_best_eval_bin = bin(idx_best_eval)[2:].zfill(n_qubit)
    print(f"\nOptimal binary string #{i+1} (evaluation):", idx_best_eval_bin)
    vec_best_eval = np.array([int(x) for x in idx_best_eval_bin])
    n_buy_eval = E @ vec_best_eval
    print("Number of each asset to buy (evaluation):", n_buy_eval)
    print("Budget used (evaluation):", round(np.sum(n_buy_eval * P), 2))
    print("Expected value (evaluation): ", round(state_optim[idx_best_eval], 7))
    print("Expected return (evaluation):", round(state_eval[idx_best_eval], 7))